# Lab 2 tasks
Name: Arhum Fareed

Student Id: FA23-BAI-006

Course: NLP


In [21]:
import PyPDF2
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
import os

# Ensure NLTK stopwords are available
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')

print("\n" + "*"*70)
print("TASK 1: PDF Document Analysis - A Student's Take")
print("*"*70)

pdf_file_path = "sample_doc.pdf"  # Check this path, please!

# Quick check if the PDF exists
if not os.path.exists(pdf_file_path):
    raise FileNotFoundError(f"Oops! The PDF file '{pdf_file_path}' is missing. Please upload 'sample_doc.pdf' or fix the path.")

# Part A: Getting text out of each page
extracted_pages_content = []
with open(pdf_file_path, "rb") as pdf_object:
    pdf_reader_instance = PyPDF2.PdfReader(pdf_object)
    for page_num_idx in range(len(pdf_reader_instance.pages)):
        current_page_text = pdf_reader_instance.pages[page_num_idx].extract_text()
        # Let's make sure we don't add empty pages, or put a placeholder
        extracted_pages_content.append(current_page_text if current_page_text else "[EMPTY PAGE]")

print(f"\n>>> Successfully pulled content from {len(extracted_pages_content)} pages.")

# Part B: Combining all text, cleaning it up a bit, and splitting into words
combined_text_source = " ".join(extracted_pages_content)
# Removing weird characters and numbers, then making everything lowercase
processed_text_for_nlp = re.sub(r'[^a-zA-Z\s]', '', combined_text_source).lower()
raw_word_tokens = processed_text_for_nlp.split()

print(f"Total raw words: {len(raw_word_tokens)}")

# Part C: Removing common words and finding the most frequent ones
english_stop_words = set(stopwords.words('english'))
meaningful_words = [individual_word for individual_word in raw_word_tokens if individual_word not in english_stop_words and len(individual_word) > 1]

word_frequency_counter = Counter(meaningful_words)
print("\nTop 10 most used words (after cleanup):")
for word, occurences in word_frequency_counter.most_common(10):
    print(f"- '{word}' showed up {occurences} times")

# Part D: Search for a specific word in sentences across pages
user_search_query = input("\nReady to search! Enter a keyword: ").strip()
query_found_on_any_page = False

print(f"Searching for: '{user_search_query}'...")
for pg_index, page_data_str in enumerate(extracted_pages_content, start=1):
    # Splitting into sentences is a bit tricky, but this is a simple way
    sentences_on_page = re.split(r'[.!?]+', page_data_str)
    for each_sentence in sentences_on_page:
        # Look for the whole word, case-insensitive
        if re.search(rf'\b{re.escape(user_search_query)}\b', each_sentence, re.IGNORECASE):
            nice_sentence = " ".join(each_sentence.split()) # Cleaning up extra spaces
            print(f"  Found on Page {pg_index}: " + nice_sentence)
            query_found_on_any_page = True

if not query_found_on_any_page:
    print(f"  Bummer! Couldn't find '{user_search_query}' anywhere. Maybe try another word?")



**********************************************************************
TASK 1: PDF Document Analysis - A Student's Take
**********************************************************************

>>> Successfully pulled content from 1 pages.
Total raw words: 607

Top 10 most used words (after cleanup):
- 'data' showed up 7 times
- 'built' showed up 7 times
- 'fine' showed up 6 times
- 'rag' showed up 5 times
- 'ai' showed up 5 times
- 'accuracy' showed up 5 times
- 'python' showed up 4 times
- 'pipelines' showed up 4 times
- 'llm' showed up 4 times
- 'pipeline' showed up 4 times

Ready to search! Enter a keyword: and
Searching for: 'and'...
  Found on Page 1: com/khairulwarahussain251203 -max PROFILE AI/ML and full -stack engineer with hands -on experience building NLP, RAG, and computer -vision systems, automated data pipelines, and production MLOps infrastructure
  Found on Page 1: Skilled at translating research and messy data into deployable tools, dashboards, and appl ications, with 

In [22]:
import PyPDF2
import re
import csv

print("\n" + "*"*70)
print("TASK 2: PDF Page Info Extraction & CSV Stuff - Another Student's Style")
print("*"*70)

# Defining a custom error type for when a search phrase isn't found, because why not?
class PhraseNotFoundMyError(Exception):
    pass

# Making sure our PDF path is still set up from Task 1
# Using the variable 'pdf_file_path' which should be defined in Task 1
# If it's not defined, this code will error out, but we're assuming Task 1 ran first!

# Let's collect some basic info for each page in the PDF
all_page_summary_data = []

# Opening the PDF and getting ready to read it page by page
with open(pdf_file_path, "rb") as my_pdf_doc:
    clever_pdf_reader = PyPDF2.PdfReader(my_pdf_doc)
    # Looping through each page with its index (starting from 1, of course)
    for page_index_here, current_page_object in enumerate(clever_pdf_reader.pages, 1):
        text_from_this_page = current_page_object.extract_text() or "" # Get text, or an empty string if none
        count_of_words = len(text_from_this_page.split()) # Easy word count
        # Character count, ignoring all whitespace characters for a cleaner number
        count_of_chars_no_spaces = len(re.sub(r'\s+', '', text_from_this_page))

        all_page_summary_data.append({
            "the_page_number": page_index_here,
            "full_page_text": text_from_this_page,
            "word_count_this_page": count_of_words,
            "char_count_no_whitespace": count_of_chars_no_spaces
        })

print("\n--- First couple of page summaries ---")
# Just showing the first two entries, keeping it brief
for entry in all_page_summary_data[:2]:
    print(f"Page {entry['the_page_number']}: {entry['word_count_this_page']} words, {entry['char_count_no_whitespace']} characters.")

# Now for some simple statistics on the word counts
# Finding the page with the most words (if there are any pages)
if all_page_summary_data:
    page_with_most_words = max(all_page_summary_data, key=lambda x: x["word_count_this_page"])
    # Filtering out pages with zero words so they don't mess up our 'min' calculation
    pages_with_actual_words = [p for p in all_page_summary_data if p["word_count_this_page"] > 0]
    page_with_fewest_words = min(pages_with_actual_words, key=lambda x: x["word_count_this_page"]) if pages_with_actual_words else None

    # Calculating the average words per page across all pages
    total_words_across_all_pages = sum(p["word_count_this_page"] for p in all_page_summary_data)
    average_words_per_page = total_words_across_all_pages / len(all_page_summary_data) if all_page_summary_data else 0

    print(f"\n-- Word Count Highlights --")
    print(f"Page {page_with_most_words['the_page_number']} had the most words: {page_with_most_words['word_count_this_page']} words.")
    print(f"(A little snippet of that page: '{page_with_most_words['full_page_text'][:100]}...')")

    if page_with_fewest_words:
        print(f"The page with the fewest words (excluding blanks) was Page {page_with_fewest_words['the_page_number']} with {page_with_fewest_words['word_count_this_page']} words.")
    print(f"On average, each page had about {average_words_per_page:.2f} words.")

# Custom Exception Search: Let's see if a phrase exists
phrase_to_look_for = input("\nWhat phrase are we looking for in the PDF? ").strip()

try:
    # Keeping track if we actually find anything
    found_anything_at_all = False
    for pg_item in all_page_summary_data:
        # Counting occurrences, case-insensitive, whole words only
        how_many_times = len(re.findall(re.escape(phrase_to_look_for), pg_item["full_page_text"], re.IGNORECASE))
        if how_many_times > 0:
            print(f"  -> Found '{phrase_to_look_for}' {how_many_times} time(s) on Page {pg_item['the_page_number']}.")
            found_anything_at_all = True

    if not found_anything_at_all:
        # If nothing was found, raise our custom error!
        raise PhraseNotFoundMyError(f"Gosh, couldn't find the phrase '{phrase_to_look_for}' anywhere! Guess it's not there.")

except PhraseNotFoundMyError as my_custom_err_msg:
    print(f"Oh no, a problem: {my_custom_err_msg}")

# Lastly, exporting our summary to a CSV file and reading it right back
csv_output_filename = "my_pdf_summary_output.csv"
with open(csv_output_filename, "w", newline="", encoding="utf-8") as csv_out_file:
    # Setting up our CSV writer and writing the column headers
    csv_writer = csv.writer(csv_out_file)
    csv_writer.writerow(["Page_Number", "Word_Count", "Character_Count"])
    for record_to_write in all_page_summary_data:
        csv_writer.writerow([
            record_to_write["the_page_number"],
            record_to_write["word_count_this_page"],
            record_to_write["char_count_no_whitespace"]
        ])

print(f"\n-- CSV Check-Up --")
print(f"Summary saved to '{csv_output_filename}'. Let's peek inside:")
# Reading the CSV file back just to confirm everything looks good
with open(csv_output_filename, "r", encoding="utf-8") as csv_in_file:
    csv_reader_for_check = csv.reader(csv_in_file)
    for a_row_from_csv in csv_reader_for_check:
        print(a_row_from_csv)



**********************************************************************
TASK 2: PDF Page Info Extraction & CSV Stuff - Another Student's Style
**********************************************************************

--- First couple of page summaries ---
Page 1: 691 words, 4129 characters.

-- Word Count Highlights --
Page 1 had the most words: 691 words.
(A little snippet of that page: 'KHAIR UL WARA HUSSAIN  
AI/ML Engineer | Python Developer  
Islamabad, Pakistan | +92 314 1839730 | ...')
The page with the fewest words (excluding blanks) was Page 1 with 691 words.
On average, each page had about 691.00 words.

What phrase are we looking for in the PDF? pipeline 
  -> Found 'pipeline' 8 time(s) on Page 1.

-- CSV Check-Up --
Summary saved to 'my_pdf_summary_output.csv'. Let's peek inside:
['Page_Number', 'Word_Count', 'Character_Count']
['1', '691', '4129']


In [24]:
# Using pre-imported stuff: `re`, `stopwords`, `pages_data` (from Task 2)

print("\n" + "*"*70)
print("TASK 3: Text Cleanup and Sentence Fun! - A Glimpse of Another Style")
print("*"*70)

# SECTION A: Let's clean up all that raw text from the PDF!
# First, grab all the text from the 'pages_data' list created in Task 2
grabbed_all_pdf_text = " ".join([p["full_page_text"] for p in all_page_summary_data]) # Using all_page_summary_data from Task 2 now
initial_char_count = len(grabbed_all_pdf_text)

# Step 1: Get rid of pesky URLs, 'cause we don't need them in our NLP stuff
text_without_urls = re.sub(r'https?://\S+|www\.\S+', '', grabbed_all_pdf_text)
# Step 2: Numbers usually aren't super useful for word analysis, so bye-bye numbers!
text_no_numbers = re.sub(r'\d+', '', text_without_urls)
# Step 3: Keep only letters, spaces, and periods for cleaner text
just_letters_and_dots = re.sub(r'[^a-zA-Z\s\.]', '', text_no_numbers)
# Step 4: Squash multiple spaces into one, because neatness counts!
final_clean_text = re.sub(r'\s+', ' ', just_letters_and_dots).strip() # Trim leading/trailing spaces too!

print(f"\nOkay, character count before all the cleaning: {initial_char_count}")
print(f"And after our cleaning spree, we have: {len(final_clean_text)} characters left!")

# SECTION B: Breaking down the clean text into individual sentences
all_lower_case_text = final_clean_text.lower()
potential_sentences = re.split(r'[.!?]', all_lower_case_text)

# Let's get rid of any empty strings or just whitespace
trimmed_sentences = [s.strip() for s in potential_sentences]
real_sentences_list = [s for s in trimmed_sentences if len(s) > 0]

print(f"\nLooks like we initially found {len(potential_sentences)} sentence-ish chunks.")
print(f"After trimming and ignoring empty ones, we've got {len(real_sentences_list)} solid sentences.")

# SECTION C: Filtering out common words and short, uninteresting sentences
common_boring_words = set(stopwords.words('english'))
super_clean_sentences = []

for a_sentence in real_sentences_list:
    # Split each sentence into words, then remove the boring ones
    sentence_tokens = a_sentence.split()
    meaningful_tokens = [a_word for a_word in sentence_tokens if a_word not in common_boring_words]

    # Only keep sentences that have at least 5 meaningful words
    if len(meaningful_tokens) >= 5:
        super_clean_sentences.append(" ".join(meaningful_tokens))

print(f"\nStarting with {len(real_sentences_list)} sentences before filtering for length.")
print(f"Now, we're down to {len(super_clean_sentences)} sentences that actually have some meat to them!")

print("\n--- Peek at the first 3 super-clean sentences ---")
for i, clean_line in enumerate(super_clean_sentences[:3]):
    print(f"{i+1}. {clean_line}")

# SECTION D: Saving our polished sentences to a file and checking it out
output_filename = "my_nlp_pipeline_output.txt"
with open(output_filename, "w", encoding="utf-8") as outfile:
    for final_sentence in super_clean_sentences:
        outfile.write(final_sentence + "\n")

print(f"\nAlright, all our cool sentences are saved to '{output_filename}'!")

# Let's quickly read it back to make sure it looks good
with open(output_filename, "r", encoding="utf-8") as infile:
    lines_read_from_file = [line.strip() for line in infile if line.strip()]

# Find the longest and shortest sentences in our file
really_long_sentence = max(lines_read_from_file, key=len) if lines_read_from_file else "(No sentences found)"
super_short_sentence = min(lines_read_from_file, key=len) if lines_read_from_file else "(No sentences found)"

print(f"We've got {len(lines_read_from_file)} lines back from the file.")
print(f"My longest sentence (whoa, {len(really_long_sentence)} characters!): '{really_long_sentence[:100]}...' ") # Showing just a snippet
print(f"My shortest sentence (just {len(super_short_sentence)} characters): '{super_short_sentence}'")



**********************************************************************
TASK 3: Text Cleanup and Sentence Fun! - A Glimpse of Another Style
**********************************************************************

Okay, character count before all the cleaning: 4943
And after our cleaning spree, we have: 4364 characters left!

Looks like we initially found 23 sentence-ish chunks.
After trimming and ignoring empty ones, we've got 23 solid sentences.

Starting with 23 sentences before filtering for length.
Now, we're down to 21 sentences that actually have some meat to them!

--- Peek at the first 3 super-clean sentences ---
1. khair ul wara hussain aiml engineer python developer islamabad pakistan khairulwarahussaingmail
2. comkhairulwarahussain max profile aiml full stack engineer hands experience building nlp rag computer vision systems automated data pipelines production mlops infrastructure
3. skilled translating research messy data deployable tools dashboards appl ications track recor

In [25]:
import docx # Need this for .docx files, obviously!
import re
from collections import Counter # Handy for word counts
import os # For checking if files exist, super important!
from pprint import pprint # Good for pretty printing dictionaries

print("\n" + "*"*70)
print("TASK 4: DOCX Document Deep Dive! - Yet Another Style")
print("*"*70)

docx_file_name = "sample_doc.docx"  # Remember to update this if your file has a different name!

# Safety check first! Wouldn't want to crash because of a missing file, right?
if not os.path.exists(docx_file_name):
    raise FileNotFoundError(f"Uh oh! The DOCX file '{docx_file_name}' isn't here. Please upload 'sample_doc.docx' or adjust the filename!")

# Let's open our document and get to work!
document_object = docx.Document(docx_file_name)

# SECTION A: Grabbing all paragraphs and their styles
all_paragraphs_data = []
for each_paragraph in document_object.paragraphs:
    all_paragraphs_data.append({
        "the_text": each_paragraph.text, # The actual words
        "its_style": each_paragraph.style.name # How it's formatted (e.g., 'Normal', 'Heading 1')
    })

# Just a quick summary of what we found
found_styles_set = set(item["its_style"] for item in all_paragraphs_data)
print(f"\nOkay, we found a total of {len(all_paragraphs_data)} paragraphs in this document.")
print(f"And these are the unique styles used: {list(found_styles_set)}")

# SECTION B: Sorting paragraphs into categories (headings, main text, other stuff)
heading_paragraphs = []
main_body_paragraphs = []
other_type_paragraphs = []

for paragraph_info in all_paragraphs_data:
    current_text = paragraph_info["the_text"].strip() # Clean up extra spaces
    current_style = paragraph_info["its_style"]

    if current_style.startswith("Heading"): # Easy way to spot headings
        heading_paragraphs.append(paragraph_info)
    elif ("Normal" in current_style or "Body" in current_style) and current_text != "": # Basic content
        main_body_paragraphs.append(paragraph_info)
    else: # Everything else goes here
        other_type_paragraphs.append(paragraph_info)

print(f"\n-- Paragraph Breakdown --")
print(f"Number of headings detected: {len(heading_paragraphs)}")
print(f"Number of main body paragraphs:     {len(main_body_paragraphs)}")
print(f"Number of other types of paragraphs:    {len(other_type_paragraphs)}")

# SECTION C: Diving deeper into body paragraphs (those with more than 5 words)
interesting_body_paragraphs = []
for a_body_para in main_body_paragraphs:
    words_in_para = a_body_para["the_text"].split() # Splitting by spaces to count words
    if len(words_in_para) > 5: # Only if it's substantial enough
        # Counting characters, ignoring all the whitespace to get a true content count
        char_count_no_whitespace = len(re.sub(r'\s+', '', a_body_para["the_text"]))
        interesting_body_paragraphs.append({
            "the_text": a_body_para["the_text"],
            "word_count": len(words_in_para),
            "char_count": char_count_no_whitespace
        })

# Let's see some stats if we have interesting paragraphs!
if interesting_body_paragraphs:
    most_words_para = max(interesting_body_paragraphs, key=lambda x: x["word_count"])
    fewest_words_para = min(interesting_body_paragraphs, key=lambda x: x["word_count"])
    avg_words_per_para = sum(p["word_count"] for p in interesting_body_paragraphs) / len(interesting_body_paragraphs)

    print("\n-- Body Paragraph Statistics --")
    print(f"Paragraph with the most words ({most_words_para['word_count']} words):\n    '{most_words_para['the_text'][:120]}...' (trimmed for brevity)")
    print(f"Paragraph with the fewest words ({fewest_words_para['word_count']} words):\n    '{fewest_words_para['the_text']}'")
    print(f"Average words per meaningful paragraph: {avg_words_per_para:.2f}")

# SECTION D: Creating a fancy dictionary to organize paragraphs by style
style_to_text_mapping = {}
for each_para_detail in all_paragraphs_data:
    style_name = each_para_detail["its_style"]
    if style_name not in style_to_text_mapping:
        style_to_text_mapping[style_name] = {"total_count": 0, "example_texts": []} # Initialize if new style
    style_to_text_mapping[style_name]["total_count"] += 1
    # We'll only keep a couple of examples to keep things neat
    if len(style_to_text_mapping[style_name]["example_texts"]) < 2:
        style_to_text_mapping[style_name]["example_texts"].append(each_para_detail["the_text"])

# Sorting for a nicer presentation (most frequent styles first)
nicely_sorted_styles = dict(sorted(style_to_text_mapping.items(), key=lambda item: item[1]["total_count"], reverse=True))

print("\n-- Style-based Paragraph Summary (Top styles shown) --")
for style, data in nicely_sorted_styles.items():
    print(f"Style: '{style}' (Used {data['total_count']} times)")
    for i, example in enumerate(data['example_texts']):
        print(f"  - Example {i+1}: '{example[:70]}...'" if len(example) > 70 else f"  - Example {i+1}: '{example}'")



**********************************************************************
TASK 4: DOCX Document Deep Dive! - Yet Another Style
**********************************************************************

Okay, we found a total of 14 paragraphs in this document.
And these are the unique styles used: ['Normal', 'List Paragraph']

-- Paragraph Breakdown --
Number of headings detected: 0
Number of main body paragraphs:     8
Number of other types of paragraphs:    6

-- Style-based Paragraph Summary (Top styles shown) --
Style: 'Normal' (Used 12 times)
  - Example 1: 'COMSATS University Islamabad'
  - Example 2: 'Islamabad Campus'
Style: 'List Paragraph' (Used 2 times)
  - Example 1: 'The student-intern is to fill this form by recording major tasks perfo...'
  - Example 2: 'Respective internship supervisors are to review and evaluate the stude...'


In [26]:
import docx # Still need this!
import pandas as pd # Essential for DataFrames, obviously!
import os # File checks are always a good idea.

print("\n" + "*"*70)
print("TASK 5: DOCX Table Treasures & Pandas Play! - A Unique Twist")
print("*"*70)

docx_file_name = "sample_doc.docx" # The same DOCX file from Task 4

# Let's make sure the file is actually there before we try to open it.
if not os.path.exists(docx_file_name):
    raise FileNotFoundError(f"Whoops! Can't find '{docx_file_name}'. Please ensure it's uploaded to Colab!")

document_for_tables = docx.Document(docx_file_name)

# SECTION A: Pulling out all the tables into a list of lists
extracted_all_tables_as_lists = []

for table_object in document_for_tables.tables:
    current_table_rows = []
    for row_object in table_object.rows:
        # Getting text from each cell in the row
        cell_texts = [individual_cell.text.strip() for individual_cell in row_object.cells]
        current_table_rows.append(cell_texts)
    extracted_all_tables_as_lists.append(current_table_rows)

print(f"\n>>> Hooray! We found {len(extracted_all_tables_as_lists)} tables in this document.")
for idx, a_table in enumerate(extracted_all_tables_as_lists, 1):
    row_count = len(a_table)
    # Make sure the table has at least one row to avoid index errors
    column_count = len(a_table[0]) if row_count > 0 else 0
    print(f"Table {idx} has {row_count} rows and {column_count} columns.")

# SECTION B: Turning the first table into a super cool Pandas DataFrame
if extracted_all_tables_as_lists:
    the_first_table_data = extracted_all_tables_as_lists[0]
    # The very first row is usually our column headers!
    table_column_names = the_first_table_data[0]
    actual_data_rows = the_first_table_data[1:]

    # Creating our DataFrame!
    table_dataframe = pd.DataFrame(actual_data_rows, columns=table_column_names)
    print("\n-- Peek at our new DataFrame from the first table --")
    print(table_dataframe.head()) # Just showing the top few rows

    # Let's check for any empty spots (None or empty strings) in our DataFrame
    # Using .applymap for older pandas versions or .map for newer ones. Sticking to map for robustness.
    empty_cells_report = table_dataframe.map(lambda x: x == "" or x is None).sum()
    print("\nEmpty cells count per column (we hate missing data!):")
    print(empty_cells_report)

# SECTION C: Grabbing all the paragraphs that appear before the first table
important_paragraphs_before_table = []

# We'll go through the document's elements until we hit a table
for doc_element in document_for_tables.element.body:
    # docx library's way of telling if it's a table element
    if doc_element.tag.endswith('tbl'):
        break # Stop when we see the first table
    # docx library's way of telling if it's a paragraph element
    if doc_element.tag.endswith('p'):
        # Convert the element back to a Paragraph object to get its text
        paragraph_content = docx.text.paragraph.Paragraph(doc_element, document_for_tables).text.strip()
        if paragraph_content: # Only add if it's not an empty paragraph
            important_paragraphs_before_table.append(paragraph_content)

print(f"\nParagraphs found just before the very first table: {len(important_paragraphs_before_table)}")

# SECTION D: Exporting everything we found to a neat text file
output_text_filename = "my_docx_analysis_output.txt"
with open(output_text_filename, "w", encoding="utf-8") as text_output_file:
    text_output_file.write("--- PARAGRAPHS BEFORE FIRST TABLE ---\n")
    for each_para in important_paragraphs_before_table:
        text_output_file.write(each_para + "\n")

    text_output_file.write("\n--- DATA FROM FIRST TABLE (CSV-like) ---\n")
    if extracted_all_tables_as_lists:
        # Write headers first
        text_output_file.write(" | ".join(the_first_table_data[0]) + "\n")
        # Then write data rows
        for data_row in the_first_table_data[1:]:
            text_output_file.write(" | ".join(data_row) + "\n")

print(f"Analysis results saved to '{output_text_filename}'! Go check it out!")



**********************************************************************
TASK 5: DOCX Table Treasures & Pandas Play! - A Unique Twist
**********************************************************************

>>> Hooray! We found 2 tables in this document.
Table 1 has 2 rows and 2 columns.
Table 2 has 7 rows and 3 columns.

-- Peek at our new DataFrame from the first table --
        Intern Name: Khair ul wara Hussain  \
0  Faculty Supervisor: Mr. Taimorr Shahzad   

                  Organization: AKSA-SDS  
0  Reporting Period: 06/07/26 – 17/08/26  

Empty cells count per column (we hate missing data!):
Intern Name: Khair ul wara Hussain    0
Organization: AKSA-SDS                0
dtype: int64

Paragraphs found just before the very first table: 4
Analysis results saved to 'my_docx_analysis_output.txt'! Go check it out!


In [27]:
import docx # Still our DOCX friend!
import re
from collections import Counter # For counting words, super useful!
import os # Always checking for files, safety first!

print("\n" + "*"*70)
print("TASK 6: Multi-Document DOCX Deep Dive & Compare! - Fresh Style")
print("*"*70)

# Defining our document names, just in case they change!
doc_alpha_file = "sample_doc.docx"
doc_beta_file = "sample_doc_2.docx"

# Gotta make sure our documents exist, or we'll have a bad time!
if not os.path.exists(doc_alpha_file):
    raise FileNotFoundError(f"Oops! Document 1, '{doc_alpha_file}', is missing. Please upload it!")

if not os.path.exists(doc_beta_file):
    raise FileNotFoundError(f"Hold up! Document 2, '{doc_beta_file}', is missing. Go upload it!")

# Let's load up our two documents
document_alpha = docx.Document(doc_alpha_file)
document_beta = docx.Document(doc_beta_file)

# SECTION A: Comparing Headings - What's common, what's unique?
# Grabbing all heading texts from Document Alpha (A)
headings_from_alpha = [
    para.text.strip() for para in document_alpha.paragraphs
    if para.style.name.startswith("Heading") and para.text.strip()
]
# Grabbing all heading texts from Document Beta (B)
headings_from_beta = [
    para.text.strip() for para in document_beta.paragraphs
    if para.style.name.startswith("Heading") and para.text.strip()
]

# Convert to sets for easy comparison (sets are great for unique items and operations!)
set_of_headings_alpha = set(headings_from_alpha)
set_of_headings_beta = set(headings_from_beta)

print("\n--- Heading Comparisons ---")
print(f"Headings found in BOTH documents: {set_of_headings_alpha.intersection(set_of_headings_beta)}")
print(f"Headings ONLY in Document A:      {set_of_headings_alpha - set_of_headings_beta}")
print(f"Headings ONLY in Document B:      {set_of_headings_beta - set_of_headings_alpha}")

# SECTION B: Merging and Deduplicating Body Paragraphs
# Getting all 'Normal' style paragraphs from Document Alpha
normal_paras_alpha = [
    para.text.strip() for para in document_alpha.paragraphs
    if "Normal" in para.style.name and para.text.strip()
]
# Getting all 'Normal' style paragraphs from Document Beta
normal_paras_beta = [
    para.text.strip() for para in document_beta.paragraphs
    if "Normal" in para.style.name and para.text.strip()
]

# Combining them all into one big list!
combined_all_body_paragraphs = normal_paras_alpha + normal_paras_beta

# Deduplication time! We want unique content, not duplicates.
unique_cleaned_body_paragraphs = []
already_seen_normalized_text = set()

for paragraph_content in combined_all_body_paragraphs:
    # Normalize the text for comparison (lowercase and single spaces)
    normalized_paragraph = " ".join(paragraph_content.lower().split())
    if normalized_paragraph not in already_seen_normalized_text:
        already_seen_normalized_text.add(normalized_paragraph)
        unique_cleaned_body_paragraphs.append(paragraph_content)

print(f"\n-- Body Paragraph Stats --")
print(f"Total 'Normal' paragraphs before removing duplicates: {len(combined_all_body_paragraphs)}")
print(f"Unique 'Normal' paragraphs after cleanup:             {len(unique_cleaned_body_paragraphs)}")

# SECTION C: Word Frequency Analysis Across Unique Body Text
# First, let's load up our English stop words (common words we want to ignore)
# Make sure to handle potential NLTK download issues if not already handled in Task 1 or 2
import nltk
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')
english_boring_words = set(nltk.corpus.stopwords.words('english'))

all_meaningful_words_collected = []

for unique_para in unique_cleaned_body_paragraphs:
    # Clean the paragraph: remove non-alphabetic characters and convert to lowercase
    cleaned_for_word_count = re.sub(r'[^a-zA-Z\s]', '', unique_para).lower()
    # Split into words, then filter out stop words and single-letter words
    individual_tokens = [
        word for word in cleaned_for_word_count.split()
        if word not in english_boring_words and len(word) > 1
    ]
    all_meaningful_words_collected.extend(individual_tokens)

# Count how often each word appears
word_occurrence_counts = Counter(all_meaningful_words_collected)

print("\n--- Top 20 Most Talked About Words ---")
for word, count_val in word_occurrence_counts.most_common(20):
    print(f"'{word}': {count_val} times")

# SECTION D: Comprehensive Summary Report - The Big Picture!
total_headings_in_both = len(headings_from_alpha) + len(headings_from_beta)
vocabulary_size = len(word_occurrence_counts)
# Calculate total words in unique body paragraphs for average length
total_words_in_unique_body = sum(len(para.split()) for para in unique_cleaned_body_paragraphs)
average_para_length = total_words_in_unique_body / len(unique_cleaned_body_paragraphs) if unique_cleaned_body_paragraphs else 0
# Find the longest unique paragraph based on word count
longest_unique_paragraph = max(unique_cleaned_body_paragraphs, key=lambda x: len(x.split())) if unique_cleaned_body_paragraphs else "(No meaningful paragraphs found)"

print("\n+---------------------------------------+")
print("|             DOCX REPORT CARD            |")
print("+---------------------------------------+")
print(f"| Total Headings (combined):        {total_headings_in_both:<5} |")
print(f"| Unique Body Paragraphs:           {len(unique_cleaned_body_paragraphs):<5} |")
print(f"| Vocabulary Size (unique words):   {vocabulary_size:<5} |")
print(f"| Average Paragraph Length:         {average_para_length:.2f} words |")
print("+---------------------------------------+")
print("\nLongest Unique Paragraph Spotlight (by word count):")
print(f"Words: {len(longest_unique_paragraph.split())}")
print(f"Text (first 150 chars): '{longest_unique_paragraph[:150]}...'\n")



**********************************************************************
TASK 6: Multi-Document DOCX Deep Dive & Compare! - Fresh Style
**********************************************************************

--- Heading Comparisons ---
Headings found in BOTH documents: set()
Headings ONLY in Document A:      set()
Headings ONLY in Document B:      {'Summer 2026', 'Student Name: Khair ul wara Hussain', 'Program: AI', 'Students’ Career Goal', 'Submitted By', 'Internship Placement Overview & Role', 'Scope of Work', 'Internship Design Proposal', 'Supervisor Details', 'Learning Objectives', 'Submission Date: 9th September 2026', 'Expected Deliverables:', 'Academic Preparation', 'COMSATS University Islamabad, Islamabad Campus', 'Key Activities (check all that apply):', 'Tools/Technologies to be used:', 'Registration No: FA23-BAI-022'}

-- Body Paragraph Stats --
Total 'Normal' paragraphs before removing duplicates: 10
Unique 'Normal' paragraphs after cleanup:             10

--- Top 20 Most T

In [29]:
import requests # Our HTTP library for talking to APIs!
from pprint import pprint # Great for neatly printing complex data

print("\n" + "*"*70)
print("TASK 7: API Data Harmonization & Insights! - A New Vibe")
print("*"*70)

# --- SECTION A: Grabbing Our Data from API Endpoints ---
# We need user details and their posts from a couple of public APIs.

print("\n--- Fetching User Data from JSONPlaceholder ---")
users_api_url = "https://jsonplaceholder.typicode.com/users"
response_users = requests.get(users_api_url)

# Always check if the API call was successful!
if response_users.status_code != 200:
    raise ConnectionError(f"Failed to retrieve users! HTTP Status: {response_users.status_code}")

all_users_raw_data = response_users.json()
print(f"Successfully retrieved {len(all_users_raw_data)} user records.")

print("\n--- Fetching Post Data from JSONPlaceholder ---")
posts_api_url = "https://jsonplaceholder.typicode.com/posts"
response_posts = requests.get(posts_api_url)

if response_posts.status_code != 200:
    raise ConnectionError(f"Failed to retrieve posts! HTTP Status: {response_posts.status_code}")

all_posts_raw_data = response_posts.json()
print(f"Successfully retrieved {len(all_posts_raw_data)} post records.")

# --- SECTION B: Merging User and Post Information ---
# Let's combine this data into a more useful structure, focusing on each user.

user_profile_with_posts = {}

for individual_user in all_users_raw_data:
    user_identification = individual_user["id"]
    user_full_name = individual_user["name"]

    # Collect all post titles for the current user
    posts_by_this_user = [
        post_entry["title"] for post_entry in all_posts_raw_data
        if post_entry["userId"] == user_identification
    ]

    user_profile_with_posts[user_full_name] = {
        "email_address": individual_user.get("email", "N/A"), # Using .get for safety
        "company_name": individual_user.get("company", {}).get("name", "N/A"),
        "total_posts_count": len(posts_by_this_user),
        "post_titles": posts_by_this_user
    }

print("\n--- Merged User-Post Data (First 3 Profiles) ---")
# Displaying a sample to keep the output concise
for user_name, user_data in list(user_profile_with_posts.items())[:3]:
    print(f"\nUser: {user_name}")
    print(f"  Email: {user_data['email_address']}")
    print(f"  Company: {user_data['company_name']}")
    print(f"  Posts: {user_data['total_posts_count']}")
    print(f"  First 2 Titles: {user_data['post_titles'][:2]}")

# --- SECTION C: Identifying the Most Prolific Author ---
# Who posts the most? Let's find out!

most_active_user_info = max(user_profile_with_posts.items(), key=lambda item: item[1]["total_posts_count"])

active_user_name = most_active_user_info[0]
active_user_details = most_active_user_info[1]

print(f"\n-- Our Top Poster Spotlight --")
print(f"The user with the most posts is: {active_user_name}")
print(f"  Email Contact: {active_user_details['email_address']}")
print(f"  Associated Company: {active_user_details['company_name']}")
print(f"  Total Posts Published: {active_user_details['total_posts_count']}")
print("  A few of their masterpieces:")
for post_idx, post_title_val in enumerate(active_user_details["post_titles"][:5], 1): # Showing top 5 titles
    print(f"    {post_idx}. {post_title_val}")

# --- SECTION D: Filtering Users Based on Specific Criteria ---
# Let's find users whose company 'bs' includes 'synergies' OR whose city name is longer than 8 characters.

print("\n--- Special Interest User Group ---")
print("Filtering users by 'synergies' in business strategy OR long city names:")

found_filtered_users = []
for user_entry in all_users_raw_data:
    company_bs_description = user_entry.get("company", {}).get("bs", "").lower()
    user_city = user_entry.get("address", {}).get("city", "")

    if "synergies" in company_bs_description or len(user_city) > 8:
        # Calculate post count for this specific user efficiently
        current_user_post_count = sum(1 for p_data in all_posts_raw_data if p_data["userId"] == user_entry["id"])
        found_filtered_users.append({
            "Name": user_entry["name"],
            "City": user_city,
            "Posts_Count": current_user_post_count
        })

if found_filtered_users:
    for f_user in found_filtered_users:
        print(f"  Name: {f_user['Name']:<20} | City: {f_user['City']:<15} | Posts: {f_user['Posts_Count']}")
else:
    print("  No users matched the filtering criteria.")



**********************************************************************
TASK 7: API Data Harmonization & Insights! - A New Vibe
**********************************************************************

--- Fetching User Data from JSONPlaceholder ---
Successfully retrieved 10 user records.

--- Fetching Post Data from JSONPlaceholder ---
Successfully retrieved 100 post records.

--- Merged User-Post Data (First 3 Profiles) ---

User: Leanne Graham
  Email: Sincere@april.biz
  Company: Romaguera-Crona
  Posts: 10
  First 2 Titles: ['sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'qui est esse']

User: Ervin Howell
  Email: Shanna@melissa.tv
  Company: Deckow-Crist
  Posts: 10
  First 2 Titles: ['et ea vero quia laudantium autem', 'in quibusdam tempore odit est dolorem']

User: Clementine Bauch
  Email: Nathan@yesenia.net
  Company: Romaguera-Jacobson
  Posts: 10
  First 2 Titles: ['asperiores ea ipsam voluptatibus modi minima quia sint', 'dolor sint quo a veli

In [30]:
# ==============================================================================
# TASK 8: To-Do Completion Rate Analysis Across Users
# ==============================================================================
print("\n" + "="*80)
print("TASK 8")
print("="*80)

# Task 8(a): Fetch and validate
res_todos = requests.get("https://jsonplaceholder.typicode.com/todos")

if res_todos.status_code != 200 or not isinstance(res_todos.json(), list) or len(res_todos.json()) == 0:
    print("Invalid response or missing data.")
    exit()

todos_list = res_todos.json()
print("Retrieved todos count:", len(todos_list))

# Task 8(b): Group per user
user_todos = {}

for t in todos_list:
    uid = t["userId"]
    if uid not in user_todos:
        user_todos[uid] = {"total": 0, "completed": 0, "incomplete": 0}

    user_todos[uid]["total"] += 1
    if t["completed"]:
        user_todos[uid]["completed"] += 1
    else:
        user_todos[uid]["incomplete"] += 1

print("\nValidation of task totals per user:")
for uid, stats in user_todos.items():
    check = (stats["completed"] + stats["incomplete"]) == stats["total"]
    print(f"User {uid}: {stats} | Total matches: {check}")

# Task 8(c): Completion rate metrics
rates = {}
for uid, stats in user_todos.items():
    pct = (stats["completed"] / stats["total"]) * 100
    rates[uid] = round(pct, 1)

best_user = max(rates.items(), key=lambda x: x[1])
worst_user = min(rates.items(), key=lambda x: x[1])
avg_completion = sum(rates.values()) / len(rates)

print(f"\nBest completion rate: User {best_user[0]} ({best_user[1]}%)")
print(f"Worst completion rate: User {worst_user[0]} ({worst_user[1]}%)")
print(f"Average rate across users: {round(avg_completion, 1)}%")

# Task 8(d): Leaderboard table
raw_users_data = requests.get("https://jsonplaceholder.typicode.com/users").json()
id_to_name = {u["id"]: u["name"] for u in raw_users_data}

table_data = []
for uid, stats in user_todos.items():
    table_data.append({
        "id": uid,
        "name": id_to_name.get(uid, f"User {uid}"),
        "total": stats["total"],
        "completed": stats["completed"],
        "rate": rates[uid]
    })

table_data.sort(key=lambda x: x["rate"], reverse=True)

print("\n" + "-"*65)
print(f"{'Rank':<6} | {'Name':<22} | {'Total':<8} | {'Done':<8} | {'Rate %':<8}")
print("-" * 65)
for rank, row in enumerate(table_data, 1):
    print(f"{rank:<6} | {row['name']:<22} | {row['total']:<8} | {row['completed']:<8} | {row['rate']:<8.1f}")
print("-" * 65)


TASK 8
Retrieved todos count: 200

Validation of task totals per user:
User 1: {'total': 20, 'completed': 11, 'incomplete': 9} | Total matches: True
User 2: {'total': 20, 'completed': 8, 'incomplete': 12} | Total matches: True
User 3: {'total': 20, 'completed': 7, 'incomplete': 13} | Total matches: True
User 4: {'total': 20, 'completed': 6, 'incomplete': 14} | Total matches: True
User 5: {'total': 20, 'completed': 12, 'incomplete': 8} | Total matches: True
User 6: {'total': 20, 'completed': 6, 'incomplete': 14} | Total matches: True
User 7: {'total': 20, 'completed': 9, 'incomplete': 11} | Total matches: True
User 8: {'total': 20, 'completed': 11, 'incomplete': 9} | Total matches: True
User 9: {'total': 20, 'completed': 8, 'incomplete': 12} | Total matches: True
User 10: {'total': 20, 'completed': 12, 'incomplete': 8} | Total matches: True

Best completion rate: User 5 (60.0%)
Worst completion rate: User 4 (30.0%)
Average rate across users: 45.0%

-------------------------------------

In [20]:
import math

print("\n" + "="*80)
print("TASK 9")
print("="*80)

# --- SECTION A: Flattening User Data ---
# The goal here is to get a simpler, flat dictionary for each user from nested JSON.
def standardize_user_entry(user_record):
    """Converts a complex user JSON object into a flattened dictionary."""
    # Note: Using .get() with default empty string for robustness against missing keys
    return {
        "id": user_record.get("id"),
        "full_name": user_record.get("name"), # Renamed from 'name'
        "user_handle": user_record.get("username"), # Renamed from 'username'
        "contact_email": user_record.get("email"), # Renamed from 'email'
        "address_city": user_record.get("address", {}).get("city"),
        "address_zip": user_record.get("address", {}).get("zipcode"),
        "latitude": float(user_record.get("address", {}).get("geo", {}).get("lat", 0.0)), # Explicit cast
        "longitude": float(user_record.get("address", {}).get("geo", {}).get("lng", 0.0)), # Explicit cast
        "employer_name": user_record.get("company", {}).get("name"),
        "employer_slogan": user_record.get("company", {}).get("catchPhrase")
    }

# Fetch original user data and apply our new flattening function
all_raw_users = requests.get("https://jsonplaceholder.typicode.com/users").json()
standardized_users_list = [standardize_user_entry(u) for u in all_raw_users]

print("\n--- User Data Standardization Output ---")
print("First simplified user record:")
# Display using standard print for a less formal look
print(standardized_users_list[0])

# --- SECTION B: Geographic Analysis with DataFrame ---
# Working with a DataFrame for easier data manipulation.
user_geo_df = pd.DataFrame(standardized_users_list)

# Calculate distance from (0,0) as a simple geographic metric
user_geo_df["euclidean_distance"] = user_geo_df.apply(
    lambda row: math.hypot(row["latitude"], row["longitude"]), axis=1 # Using math.hypot for clarity
)

# Find the user furthest from the 'origin'
farthest_traveler = user_geo_df.loc[user_geo_df["euclidean_distance"].idxmax()]
print(f"\n-- Geographic Distance Stats --")
print(f"The user furthest from the center is: {farthest_traveler['full_name']} (Distance: {farthest_traveler['euclidean_distance']:.2f})")

# --- SECTION C: Hemisphere Grouping and Averages ---
# Dividing users into northern and southern hemispheres based on latitude.
northern_hemi_folks = user_geo_df[user_geo_df["latitude"] > 0]
southern_hemi_folks = user_geo_df[user_geo_df["latitude"] < 0]

print("\n-- Hemisphere Breakdown --")
print("Users in Northern Hemisphere:")
for name in northern_hemi_folks["full_name"]:
    print(f"- {name}")

print("\nUsers in Southern Hemisphere:")
for name in southern_hemi_folks["full_name"]:
    print(f"- {name}")

# Calculate average distances for comparison
avg_north_dist = northern_hemi_folks["euclidean_distance"].mean()
avg_south_dist = southern_hemi_folks["euclidean_distance"].mean()

print(f"\nAverage distance (Northern): {avg_north_dist:.2f}")
print(f"Average distance (Southern): {avg_south_dist:.2f}")

if avg_north_dist > avg_south_dist:
    print("Observation: Northern hemisphere users are, on average, farther out.")
else:
    print("Observation: Southern hemisphere users are, on average, farther out.")

# --- SECTION D: Post and Comment Engagement Analysis ---
# Fetching posts and comments to analyze engagement.
all_posts_data = requests.get("https://jsonplaceholder.typicode.com/posts").json()
all_comments_data = requests.get("https://jsonplaceholder.typicode.com/comments").json()

# Count comments per post using a dictionary for cleaner access
comment_counts_by_postid = {post_id: 0 for post_id in set(c['postId'] for c in all_comments_data)}
for comment in all_comments_data:
    comment_counts_by_postid[comment['postId']] += 1

# Compile a list of posts with their comment counts
post_engagement_summary = []
for p_item in all_posts_data:
    post_id_val = p_item["id"]
    brief_title = p_item["title"][:50] # Take a slightly longer title snippet
    num_comments = comment_counts_by_postid.get(post_id_val, 0)
    post_engagement_summary.append({"id": post_id_val, "title_snippet": brief_title, "comment_total": num_comments})

# Sort posts by comment count for leaderboard style output
post_engagement_summary.sort(key=lambda x: x["comment_total"], reverse=True)

print("\n-- Post Engagement Report --")
print("Top 5 Most Discussed Posts:")
for i, post in enumerate(post_engagement_summary[:5]):
    print(f"{i+1}. ID: {post['id']:<4} | Title: {post['title_snippet']:<50} | Comments: {post['comment_total']}")

print("\nBottom 5 Least Discussed Posts:")
# Note: This assumes there are at least 5 posts. Add a check if needed for real data.
for i, post in enumerate(post_engagement_summary[-5:]):
    print(f"{len(post_engagement_summary)-4+i}. ID: {post['id']:<4} | Title: {post['title_snippet']:<50} | Comments: {post['comment_total']}")


TASK 9

--- User Data Standardization Output ---
First simplified user record:
{'id': 1, 'full_name': 'Leanne Graham', 'user_handle': 'Bret', 'contact_email': 'Sincere@april.biz', 'address_city': 'Gwenborough', 'address_zip': '92998-3874', 'latitude': -37.3159, 'longitude': 81.1496, 'employer_name': 'Romaguera-Crona', 'employer_slogan': 'Multi-layered client-server neural-net'}

-- Geographic Distance Stats --
The user furthest from the center is: Glenna Reichert (Distance: 170.68)

-- Hemisphere Breakdown --
Users in Northern Hemisphere:
- Patricia Lebsack
- Kurtis Weissnat
- Glenna Reichert

Users in Southern Hemisphere:
- Leanne Graham
- Ervin Howell
- Clementine Bauch
- Chelsey Dietrich
- Mrs. Dennis Schulist
- Nicholas Runolfsdottir V
- Clementina DuBuque

Average distance (Northern): 123.58
Average distance (Southern): 84.32
Observation: Northern hemisphere users are, on average, farther out.

-- Post Engagement Report --
Top 5 Most Discussed Posts:
1. ID: 1    | Title: sunt aut